# SSI FastConnect — kiểm tra OHLCV cho TradingAgents

Notebook này dùng Python SDK chính thức `ssi-sdk` thay cho đoạn tự gọi REST endpoint bị `HTTP 400`. Chỉ đọc OHLCV của FPT; không có chức năng đặt lệnh.

### Cách chạy
1. Mở bằng Google Colab và chọn **Runtime → Run all**.
2. Để khỏi nhập lại credentials mỗi lần, lưu chúng trong Colab **Secrets** với tên `SSI_CLIENT_ID`, `SSI_API_KEY`, `SSI_API_SECRET`. Notebook cũng tái sử dụng các biến `CLIENT_ID`, `API_KEY`, `API_SECRET` nếu đã tồn tại trong cùng runtime.
3. Credentials chỉ được dùng trong runtime; không thêm chúng vào GitHub hoặc gửi qua chat.

Nguồn API: [SSI OHLC tutorial](https://developers.ssi.com.vn/docs/tutorials/ohlc), [SSI SDK services](https://developers.ssi.com.vn/docs/sdk/python/service-classes).

In [ ]:
%pip -q install "ssi-sdk==3.3.2" pandas

In [ ]:
import inspect
import ssi_sdk
from ssi_sdk import Auth, Data, Config
from ssi_sdk.enums import Timeframe
from ssi_sdk.services.market_data import MarketDataService

# Contract checks only; this does NOT make a network request or use credentials.
assert ssi_sdk.__version__ == "3.3.2", f"Unexpected SSI SDK version: {ssi_sdk.__version__}"
assert callable(getattr(MarketDataService, "get_ohlc_1day_historical", None))
assert callable(getattr(MarketDataService, "get_ohlc_all", None))
params = inspect.signature(MarketDataService.get_ohlc_all).parameters
for required in ("symbol", "timeframe", "from_date", "to_date", "size", "ascending"):
    assert required in params, f"SSI SDK method signature changed: missing {required}"
assert Timeframe.DAY_1.value == "1d", f"Unexpected day timeframe: {Timeframe.DAY_1}"
probe = Config(client_id="cid", api_key="key123", api_secret="sec123")
assert "sec123" not in repr(probe), "SDK Config repr unexpectedly exposes the API secret"
print(f"✅ SSI SDK preflight passed: version={ssi_sdk.__version__}; method signatures and secret masking verified.")
print("Chưa gọi SSI thật ở bước này; xác thực live sẽ diễn ra ở ô tiếp theo.")

In [ ]:
import os
import re
from getpass import getpass

def _colab_secret(name):
    """Read a Colab Secret if available; never print its value."""
    value = os.environ.get(name)
    if value:
        return value.strip()
    try:
        from google.colab import userdata
        value = userdata.get(name)
        return value.strip() if value else None
    except Exception:
        return None

def _credential(name, prompt, legacy_names=()):
    value = _colab_secret(name)
    if not value:
        # Reuse credentials already present if this cell is run in an existing notebook runtime.
        for legacy_name in legacy_names:
            candidate = globals().get(legacy_name)
            if isinstance(candidate, str) and candidate.strip():
                value = candidate.strip()
                break
    if value:
        print(f"{name}: đã nạp sẵn (giá trị được ẩn).")
        return value
    print(f"Chưa tìm thấy {name}. Nhập một lần, hoặc lưu vào Colab Secrets để dùng lại.")
    return getpass(prompt).strip()

CLIENT_ID = _credential("SSI_CLIENT_ID", "SSI Client ID (không hiển thị): ", ("CLIENT_ID",))
API_KEY = _credential("SSI_API_KEY", "SSI API Key (không hiển thị): ", ("API_KEY",))
API_SECRET = _credential("SSI_API_SECRET", "SSI API Secret (không hiển thị): ", ("API_SECRET",))

if not all((CLIENT_ID, API_KEY, API_SECRET)):
    raise ValueError("Thiếu Client ID/API Key/API Secret. Kiểm tra Colab Secrets hoặc nhập lại.")

print("Đã nhận đủ thông tin xác thực. Không có credential nào được in ra.")

In [ ]:
from datetime import date, timedelta
import re
import pandas as pd

SYMBOL = "FPT"
TO_DATE = date.today().strftime("%Y/%m/%d") + " 23:59:59"
FROM_DATE = (date.today() - timedelta(days=60)).strftime("%Y/%m/%d") + " 00:00:00"

config = Config(
    client_id=CLIENT_ID,
    api_key=API_KEY,
    api_secret=API_SECRET,
    # private_key chỉ cần cho các thao tác ký lệnh; ô này chỉ đọc dữ liệu thị trường.
)

def _safe_exception_message(exc):
    """Use the SDK's public error message, removing credentials/JWTs before displaying it."""
    message = getattr(exc, "message", None) or type(exc).__name__
    message = str(message)
    for secret in (CLIENT_ID, API_KEY, API_SECRET):
        if secret:
            message = message.replace(secret, "[REDACTED]")
    message = re.sub(r"(?i)(authorization\s*[:=]\s*bearer\s+)[^\s,;]+", r"\1[REDACTED]", message)
    message = re.sub(r"eyJ[A-Za-z0-9_-]{15,}\.[A-Za-z0-9_-]{10,}\.[A-Za-z0-9_-]+", "[REDACTED_TOKEN]", message)
    return message[:800]

print(f"Đang đọc OHLC ngày cho {SYMBOL}, từ {FROM_DATE} đến {TO_DATE}...")
try:
    with Auth(config) as auth:
        auth.authenticate()  # Theo SSI SDK, Market Data không yêu cầu OTP.
        with Data(auth) as data:
            # Lấy đủ các trang và yêu cầu nến theo thứ tự cũ → mới.
            bars = data.market_data.get_ohlc_all(
                SYMBOL,
                Timeframe.DAY_1,
                from_date=FROM_DATE,
                to_date=TO_DATE,
                size=100,
                ascending=True,
            )
except Exception as exc:
    status = getattr(exc, "status_code", None)
    code = getattr(exc, "code", None)
    detail = _safe_exception_message(exc)
    print(f"SSI Market Data lỗi: type={type(exc).__name__}, HTTP={status}, code={code}")
    print(f"Thông báo đã lọc thông tin nhạy cảm: {detail}")
    raise RuntimeError("Không lấy được OHLCV qua SSI SDK; xem loại lỗi và thông báo an toàn ngay phía trên.") from None

if not isinstance(bars, list):
    raise TypeError(f"SDK contract lỗi: kỳ vọng list[OHLCData], nhận {type(bars).__name__}")
print(f"✅ SSI SDK trả về {len(bars)} nến cho {SYMBOL}.")

In [ ]:
def _field(obj, name):
    return obj.get(name) if isinstance(obj, dict) else getattr(obj, name, None)

columns = ["time", "symbol", "open", "high", "low", "close", "volume", "value"]
records = []
for bar in bars:
    records.append({
        "time": _field(bar, "trading_date"),
        "symbol": _field(bar, "symbol") or SYMBOL,
        "open": _field(bar, "open_price"),
        "high": _field(bar, "high_price"),
        "low": _field(bar, "low_price"),
        "close": _field(bar, "close_price"),
        "volume": _field(bar, "volume"),
        "value": _field(bar, "value"),
    })

ohlcv_df = pd.DataFrame(records, columns=columns)
if ohlcv_df.empty:
    print("⚠️ SSI trả list rỗng. Xác thực/call đã không ném lỗi nhưng chưa có nến trong khoảng; kiểm tra phiên giao dịch và quyền dữ liệu.")
else:
    ohlcv_df["time"] = pd.to_datetime(ohlcv_df["time"], errors="coerce")
    if ohlcv_df["time"].isna().any():
        bad_count = int(ohlcv_df["time"].isna().sum())
        raise ValueError(f"Có {bad_count} bản ghi OHLC có trading_date không parse được; không xuất dữ liệu sai âm thầm.")
    ohlcv_df = ohlcv_df.sort_values("time").reset_index(drop=True)
    if ohlcv_df["time"].duplicated().any():
        raise ValueError("Có ngày giao dịch trùng lặp trong dữ liệu OHLC; cần kiểm tra dữ liệu nguồn.")
    display(ohlcv_df.tail(20))
    print(f"✅ OHLCV hợp lệ: {len(ohlcv_df)} nến, {ohlcv_df['time'].min().date()} → {ohlcv_df['time'].max().date()}.")